# JSX 与 TSX

学习目标：理解 TSX 的类型来源和转换入口，并用自带最小运行实现完成检查、转换与执行。

前置知识：函数、对象类型、泛型、包的条件导出与 ES 模块。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；strict、react-jsx，使用本章最小运行实现。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/26-jsx-tsx/。

1. [main.tsx](scripts/26-jsx-tsx/main.tsx)：TSX 组件及描述树输出。
2. [runtime/jsx-runtime.ts](scripts/26-jsx-tsx/runtime/jsx-runtime.ts)：描述树类型、JSX 声明和 jsx/jsxs 实现。
3. [package.json](scripts/26-jsx-tsx/package.json)：自动运行时入口的本地包自引用。
4. [tsconfig.json](scripts/26-jsx-tsx/tsconfig.json)：react-jsx 编译；保留语法配置为 tsconfig.preserve.json。
5. [type-errors.tsx](scripts/26-jsx-tsx/type-errors.tsx)：标签、属性、children 和返回类型反例。

## 1 JSX 是需要转换的语法

JSX 是可嵌入表达式的标签语法，不是浏览器原生 HTML。TypeScript 在 .tsx 文件中解析并检查 JSX；输出取决于 jsx 选项，具体含义由运行实现提供。

本章自带描述树运行实现，不使用框架。它接收字符串子节点与树节点，输出可观察的描述，不创建 DOM，也不实现状态更新、事件、key 协调或 Fragment。TSX 中类型断言使用 as，避免尖括号断言和标签解析冲突。

以下片段来自 main.tsx。

```tsx
import { describe, type JSX, type Child } from "./runtime/jsx-runtime.js";
function Badge(props: { title: string; children?: Child }): JSX.Element {
  return <section><span tone="quiet">{props.title}</span>{props.children ?? "空"}</section>;
}
const view = <Badge title="Ada">ready</Badge>;
console.log(describe(view)); // section(span(Ada),ready)：本例生成描述树。
```

Step 1：检查本章正常项目。

```bash
npm run check:26
# 无类型诊断。
```

Step 2：生成当前源码的 JavaScript。

```bash
npm run build:26
# 类型错误时不生成新输出。
```

Step 3：执行本章运行入口。

```bash
npm run run:26
# 输出 section(span(Ada),ready)，与描述树的节点顺序一致。
```

## 2 编译模式和运行入口

先分清“谁检查标签”“谁转换语法”和“谁真正创建结果”，再选择 JSX 编译模式。

jsx 决定怎样转换，jsxImportSource 指定自动运行时的类型和实现来自哪个模块，不会安装运行库。

| 模式名称 | 中文名称／含义 | 输出形态 |
| --- | --- | --- |
| preserve | 保留 JSX | .jsx，留给后续工具转换 |
| react | 经典工厂调用 | 默认 React.createElement，可配置工厂 |
| react-jsx | 自动运行时调用 | 导入 jsx 或 jsxs，输出 .js |
| react-jsxdev | 开发自动运行时调用 | jsxDEV 与开发信息 |
| react-native | 在 .js 中保留 JSX | 后续工具处理 |

本例只执行 react-jsx；开发模式需要另一个 jsx-dev-runtime 入口，不能把切换选项等同于已经提供实现。

![TSX 从标签语法到本例描述树。本章执行 react-jsx 自动运行时模式；输出不创建 DOM。](image/illustration/26-01-tsx-transform-runtime.svg)

图示说明（依据篇末官方文档自绘）：图是本例的描述树流程，不代表浏览器渲染管线或任意框架的内部实现。

对照下方 jsx、jsxImportSource 和 exports 配置，再查看生成文件中的导入与 jsx/jsxs 调用。

以下片段来自 tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [
      "node"
    ],
    "rootDir": ".",
    "outDir": "./.build",
    "noEmitOnError": true,
    "jsx": "react-jsx",
    "jsxImportSource": "notebook-jsx-runtime-ts-c"
  },
  "files": [
    "main.tsx",
    "runtime/jsx-runtime.ts"
  ]
}
```

自动转换会导入 notebook-jsx-runtime-ts-c/jsx-runtime。通过 exports 映射到真实文件，避免给 Node.js 的 ESM 加载留下没有扩展名的相对导入。本目录的包名允许包内自引用，无需下载同名包。types 指向开发源码，default 指向生成的运行文件；这是章节内部开发配置，不是发布包验收。

以下片段来自 package.json。

```json
{
  "name": "notebook-jsx-runtime-ts-c",
  "private": true,
  "type": "module",
  "exports": {
    "./jsx-runtime": {
      "types": "./runtime/jsx-runtime.ts",
      "default": "./.build/runtime/jsx-runtime.js"
    }
  }
}
```

## 3 JSX 命名空间与标签检查

自动运行时从 jsx-runtime 导出的 JSX 命名空间读取类型。小写标签属于 intrinsic elements（本例的内置标签），由 JSX.IntrinsicElements 描述；其每个属性声明一个标签及其允许的属性。本例只接受 section 和 span，不添加允许任意标签的索引签名。

Badge 是基于值的元素（value-based element），从当前作用域找到函数，其第一个参数决定属性类型。JSX.Element 规定 JSX 表达式的结果，本例是 View 描述树；它不保留每个标签的全部具体类型信息。ElementChildrenAttribute 指定嵌套内容进入 children 属性。

TypeScript 5.1 起可用 JSX.ElementType 指定哪些类型能作为标签。本例限定为 IntrinsicElements 的键或返回 View 的函数，匹配最小运行实现。只定义 Element 仍会沿用允许 null 返回值和类组件的默认检查，不能据此推断运行支持范围。

ElementType 中的 props: never 只用于类型匹配：strict 下函数参数按逆变关系检查，never 可赋给不同的属性类型，因此这些组件无需共用同一种属性结构。它不是实际调用参数；JSX 中的属性仍按组件第一个参数检查。

View 表示树节点，Child 表示树节点或字符串。children 的允许范围由每个标签或组件的属性类型决定，不是只要写了嵌套内容就一律通过。

以下片段来自 runtime/jsx-runtime.ts。

```typescript
export interface View { tag: string; children: Child[] }
export type Child = View | string;
export namespace JSX {
  export type ElementType = keyof IntrinsicElements | ((props: never) => View);
  export interface Element extends View {}
  export interface ElementChildrenAttribute { children: {} }
  export interface IntrinsicElements {
    section: { children?: Child | Child[] };
    span: { tone?: "quiet" | "loud"; children?: Child | Child[] };
  }
}
```

## 4 最小运行实现与实际转换

类型声明不构造树。jsx 真正区分标签字符串和组件函数：字符串创建 View，函数则被调用以取得 View。P 表示本次调用的属性对象类型，children 被规范成数组。jsxs 在本例复用该逻辑，以接收多子节点调用。

describe 递归生成稳定字符串，仅用于观察树，不把内容当成 HTML。构建后的 main.js 导入 jsx/jsxs 并改写标签调用，类型导入被擦除，Node.js 运行这个生成物。

```typescript
export function jsx<P extends { children?: Child | Child[] }>(
  tag: string | ((props: P) => View), props: P
): View {
  if (typeof tag === "function") return tag(props);
  const children = props.children;
  return { tag, children: children === undefined ? [] : Array.isArray(children) ? children : [children] };
}
export const jsxs = jsx;
export function describe(view: View): string {
  return view.tag + "(" + view.children.map(child => typeof child === "string" ? child : describe(child)).join(",") + ")";
}
```

Step 1：生成保留 JSX 的对照产物。

```bash
npm run preserve:26
# .preserve/main.jsx 仍含 JSX，不是本例 Node.js 的运行入口。
```

## 5 属性、children 与组件返回类型

缺少 title、tone 的字面量错误和 children 类型错误分别违反不同的属性契约。反例中的 Badge 把 children 限制成 string，因此嵌套 span 被拒绝；正常 Badge 则允许 Child。

Bad 和 Empty 分别返回 number、null，不满足 ElementType 中的 View 返回要求。ClassView 的实例虽有 View 的字段，类本身只有构造签名，不能被 jsx 当普通函数调用，因此也被拒绝。这是本例的范围，其他运行实现可声明不同的合法组件类型。

以下片段来自 type-errors.tsx。

```tsx
import type { JSX, Child } from "./runtime/jsx-runtime.js";
function Badge(props: { title: string; children?: string }): JSX.Element {
  return <span>{props.title}</span>;
}
const missing = <Badge />; // TS2741：缺 title。
const invalid = <span tone="bright" />; // TS2322：属性字面量不匹配。
const children = <Badge title="a"><span /></Badge>; // TS2322：children 要求 string。
const unknown = <unknownTag />; // TS2339、TS2786：未声明的内置标签。
function Bad() { return 42; }
const returnType = <Bad />; // TS2786：返回 number 不满足 ElementType 的 View 返回要求。
function Empty() { return null; }
const empty = <Empty />; // TS2786：返回 null 不满足 View 返回要求。
class ClassView { tag = "span"; children: Child[] = []; }
const classView = <ClassView />; // TS2786：类只有构造签名，不满足本例的调用签名。
```

Step 1：检查独立 TSX 反例。

```bash
npm run errors:26
# 退出 1；包含 TS2741、TS2322、TS2339、TS2786；Empty、ClassView 均报 TS2786。
```

## 本章小结

- TSX 语法、JSX 类型和运行实现需要匹配。
- ElementType 限制合法标签，Element 描述表达式结果；实际属性仍单独检查。
- jsxImportSource 选择入口，类型通过不意味着运行实现已存在。

## 练习

1. 增加 strong 标签及其属性类型，核对 describe 包含 strong(Ada)，拼错标签仍报错。
2. 把 Badge 的 children 限制为 string：文本通过，树节点产生 TS2322。
3. 对照 .build/main.js 和 .preserve/main.jsx，找到转换调用与保留标签，说明为何只有前者能由本例 Node.js 入口执行。

### 提示

1. 在 IntrinsicElements 增加 strong 的属性类型，再替换 Badge 内的 span。
2. 使用独立反例观察不匹配 children，避免让正常构建混入故意错误。
3. 构建两种产物后逐项查看相同表达式的位置。

### 参考解析

1. 可增加 `strong: { children?: Child | Child[] }`，并把内部 span 改为 `&lt;strong&gt;{props.title}&lt;/strong&gt;`。字符串标签运行逻辑无需增加专门分支，描述应为 section(strong(Ada),ready)。
2. children 为 string 时，文本 ready 合法，树节点不再可赋值给 string。属性检查依据组件参数类型，不能只改变 Child 别名而忽略其他使用者。
3. react-jsx 产物导入 jsx/jsxs 并调用它们；preserve 产物仍含标签语法，需要后续转换，不能直接当本例的 Node JavaScript 入口。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [JSX](https://www.typescriptlang.org/docs/handbook/jsx.html) 的 Basic usage、JSX namespace、Intrinsic elements、Value-based elements、Children Type Checking、JSX result type、JSX function return type；[5.1 标签类型与表达式类型解耦](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-1.html#decoupled-type-checking-between-jsx-elements-and-jsx-tag-types)：模式、属性及合法组件范围；[严格函数类型](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-2-6.html#strict-function-types)、[never 的可赋值性](https://www.typescriptlang.org/docs/handbook/2/narrowing.html#the-never-type)：本例 ElementType 参数位置的类型匹配依据。 |
| Node.js 24.11.0 | [包自引用](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html#self-referencing-a-package-using-its-name)、[条件导出](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html#conditional-exports)：本地运行时入口解析。 |
